# Polynomial regression and feature engineering

A straight line cannot follow a curve. The fix is not a new algorithm. We keep linear regression and give it better columns. This notebook shows how, and uses the same trick to show underfitting and overfitting in their purest form.

Order: intuition, maths, cost function, from-scratch version, library version, tuning, what breaks.

In [ ]:
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
from sklearn.linear_model import LinearRegression, Ridge
from sklearn.metrics import mean_squared_error
from sklearn.model_selection import GridSearchCV, KFold, cross_val_score
from sklearn.pipeline import make_pipeline
from sklearn.preprocessing import PolynomialFeatures, StandardScaler

from common.datasets import load_california_housing, make_noisy_curve, true_curve

SEED = 42
rng = np.random.default_rng(SEED)
np.set_printoptions(precision=4, suppress=True)
plt.rcParams["figure.figsize"] = (6, 3.5)

## 1. Intuition

"Linear" in linear regression means linear in the weights, not in the inputs. The model $w_0+w_1x+w_2x^2$ bends, yet it is still a weighted sum of columns. The columns are $1$, $x$ and $x^2$.

So the recipe is: invent new columns from the old ones, then run ordinary linear regression on the lot. Squares, products of two features, logs and ratios are all fair game. Choosing them is feature engineering, and a good column often beats a cleverer algorithm.

The risk is flexibility. Each extra column is another knob, and enough knobs fit the noise as well as the signal.

## 2. Maths

For one input $x$ and degree $d$:

$$\hat y=w_0+w_1x+w_2x^2+\dots+w_dx^d=\boldsymbol\phi(x)^\top\mathbf w,\qquad \boldsymbol\phi(x)=(1,x,x^2,\dots,x^d)$$

The cost is the same MSE as before, and so is the solution, with $X$ replaced by the matrix $\Phi$ of new columns:

$$\hat{\mathbf w}=(\Phi^\top\Phi)^{-1}\Phi^\top\mathbf y$$

With $p$ original features, all terms up to degree $d$ number $\binom{p+d}{d}$ (without the constant, one fewer). That grows quickly. Ten features at degree 3 is 285 columns.

In [ ]:
from math import comb

for p in (1, 2, 8, 20):
    print(f"p={p:>2}:", {d: comb(p + d, d) - 1 for d in (2, 3, 5)})

## 3. Feature engineering on real data

On the housing data, `AveRooms` and `AveBedrms` are per-household averages and `Population` is heavily skewed. A few engineered columns help a linear model. I score with 5-fold cross-validated $R^2$ on a 2,000-row sample, with the scaler inside the pipeline.

In [ ]:
X, y = load_california_housing(n_samples=2000, seed=SEED)
cv = KFold(5, shuffle=True, random_state=SEED)


def cv_r2(features: pd.DataFrame, model) -> float:
    return float(cross_val_score(model, features, y, cv=cv, scoring="r2").mean())


base = make_pipeline(StandardScaler(), LinearRegression())
# Cap extreme values at fixed, sensible limits (a few blocks have 1,000+ rooms per house)
caps = {"AveRooms": 10, "AveBedrms": 3, "AveOccup": 6, "Population": 5000}
X = X.assign(**{c: X[c].clip(upper=v) for c, v in caps.items()})
print("original 8 features, capped:", cv_r2(X, base))

X_fe = X.copy()
X_fe["log_pop"] = np.log1p(X["Population"])
X_fe["log_occup"] = np.log(X["AveOccup"].clip(lower=0.5))
X_fe["bedroom_ratio"] = X["AveBedrms"] / X["AveRooms"]
X_fe["log_income"] = np.log(X["MedInc"])
print("with 4 engineered  :", cv_r2(X_fe, base))

Capping a few absurd values does most of the work and the four columns add a little on top. The cap matters more than it looks, because powers amplify outliers. The cell after the next one shows the damage. Now interactions and squares through `PolynomialFeatures`. Degree 2 on 8 features gives 44 columns, degree 3 gives 164.

In [ ]:
for d in (1, 2, 3):
    model = make_pipeline(
        PolynomialFeatures(d, include_bias=False), StandardScaler(), LinearRegression()
    )
    n_cols = PolynomialFeatures(d, include_bias=False).fit(X).n_output_features_
    print(f"degree {d}: {n_cols:>3} columns, CV R^2 = {cv_r2(X, model):.3f}")

Degree 2 helps and degree 3 falls back, because 164 columns on 1,600 training rows per fold is too many. The cross-validated score is what tells you when you have gone too far.

Now the same degree-2 model without the cap. One block with 1,000+ rooms per house becomes a million once squared.

In [ ]:
X_raw, _ = load_california_housing(n_samples=2000, seed=SEED)
m2 = make_pipeline(PolynomialFeatures(2, include_bias=False), StandardScaler(), LinearRegression())
print("degree 2, uncapped CV R^2:", cv_r2(X_raw, m2))

A strongly negative $R^2$ means worse than predicting the mean. Look at your data for extremes before you raise features to powers.

## 4. A clean overfitting demo

Real data is messy, so for the main demo I use a curve I control: $y=\sin(1.5x)+x/2$ plus noise, with only 30 training points. I also draw a large separate test set from the same curve. The generator comes from `common.datasets`.

In [ ]:
X_tr, y_tr = make_noisy_curve(n_samples=30, noise=0.3, seed=1)
X_te, y_te = make_noisy_curve(n_samples=300, noise=0.3, seed=99)
grid = np.linspace(-3, 3, 300)
plt.scatter(X_tr["x"], y_tr, label="train")
plt.plot(grid, true_curve(grid), c="k", label="true curve")
plt.legend()
plt.show()

### From scratch

Build the powers of $x$ with `np.vander` and solve least squares. This is a few lines on top of notebook 10.

In [ ]:
def fit_poly(x: np.ndarray, target: np.ndarray, degree: int) -> np.ndarray:
    Phi = np.vander(x, degree + 1, increasing=True)  # columns 1, x, x^2, ...
    return np.linalg.lstsq(Phi, target, rcond=None)[0]


def predict_poly(w: np.ndarray, x: np.ndarray) -> np.ndarray:
    return np.vander(x, len(w), increasing=True) @ w


w3 = fit_poly(X_tr["x"].to_numpy(), y_tr.to_numpy(), 3)
print("degree 3 weights:", w3)
print("train MSE:", mean_squared_error(y_tr, predict_poly(w3, X_tr["x"].to_numpy())))

### Library version, across degrees

In practice I use a pipeline: generate powers, scale them (powers of $x$ differ enormously in size), then fit. Plot four degrees.

In [ ]:
def poly_model(degree: int, estimator=None):
    est = estimator if estimator is not None else LinearRegression()
    return make_pipeline(PolynomialFeatures(degree, include_bias=False), StandardScaler(), est)


fig, axes = plt.subplots(1, 4, figsize=(14, 3), sharey=True)
for ax, d in zip(axes, (1, 3, 9, 15)):
    m = poly_model(d).fit(X_tr, y_tr)
    ax.scatter(X_tr["x"], y_tr, s=10)
    ax.plot(grid, true_curve(grid), c="k", lw=1)
    ax.plot(grid, m.predict(pd.DataFrame({"x": grid})), c="r")
    ax.set_ylim(-4, 4)
    ax.set_title(f"degree {d}")
plt.show()

Degree 1 is a line through a wave: it underfits (high bias). Degree 3 follows the shape. Degree 9 and 15 chase individual points and swing wildly between them: they overfit (high variance). Now measure it. Train error keeps falling. Test error is the honest one.

In [ ]:
degrees = range(1, 16)
tr_err, te_err = [], []
for d in degrees:
    m = poly_model(d).fit(X_tr, y_tr)
    tr_err.append(mean_squared_error(y_tr, m.predict(X_tr)))
    te_err.append(mean_squared_error(y_te, m.predict(X_te)))

plt.plot(degrees, tr_err, label="train")
plt.plot(degrees, te_err, label="test")
plt.axhline(0.3**2, ls="--", c="grey", label="noise floor (0.09)")
plt.yscale("log")
plt.xlabel("degree")
plt.ylabel("MSE")
plt.legend()
plt.show()
print("best test degree:", list(degrees)[int(np.argmin(te_err))])
print("degree 15 train vs test:", tr_err[-1], te_err[-1])

No model can beat the noise floor of $0.3^2=0.09$, because that part of $y$ is random. Training error can dip below it, which means the model is fitting noise. That is the clearest sign of overfitting there is.

The weights tell the same story. The overfit models need huge weights that cancel each other out.

In [ ]:
for d in (3, 9, 15):
    w = poly_model(d).fit(X_tr, y_tr)[-1].coef_
    print(f"degree {d:>2}: largest |weight| = {np.abs(w).max():.1f}")

## 5. Tuning the degree

The degree is a hyperparameter. Choose it by cross-validation on the training set, never on the test set. `GridSearchCV` does the loop and refits the best model.

In [ ]:
search = GridSearchCV(
    poly_model(1),
    {"polynomialfeatures__degree": list(range(1, 13))},
    cv=KFold(5, shuffle=True, random_state=SEED),
    scoring="neg_mean_squared_error",
).fit(X_tr, y_tr)

print("CV picks degree:", search.best_params_["polynomialfeatures__degree"])
print("CV MSE:", -search.best_score_)
print("test MSE:", mean_squared_error(y_te, search.predict(X_te)))

Cross-validation, which never saw the test set, lands on a low degree and a test error near the noise floor.

## 6. Regularisation, a preview

Instead of picking a small degree, keep a flexible model and penalise large weights. Ridge regression adds $\alpha\sum w_j^2$ to the cost. Same degree 15, with and without it:

In [ ]:
plain = poly_model(15).fit(X_tr, y_tr)
ridge = poly_model(15, Ridge(alpha=1.0)).fit(X_tr, y_tr)
for name, m in [("plain", plain), ("ridge", ridge)]:
    print(
        f"{name:>5}: train {mean_squared_error(y_tr, m.predict(X_tr)):.3f}  test {mean_squared_error(y_te, m.predict(X_te)):.3f}"
    )

The penalty gives up a bit of training fit and gets much of the test error back. Notebook 13 covers this properly.

## 7. What breaks

**Extrapolation.** A polynomial is only trustworthy where there is data. Outside $[-3,3]$ high degrees fly off.

In [ ]:
far = np.linspace(-4.5, 4.5, 200)
for d in (1, 3, 15):
    m = poly_model(d).fit(X_tr, y_tr)
    p = m.predict(pd.DataFrame({"x": np.array([-4.5, 4.5])}))
    print(
        f"degree {d:>2}: prediction at x=-4.5 and 4.5 -> {p}   (true {true_curve(np.array([-4.5, 4.5]))})"
    )

**Numerical trouble.** Raw powers of $x$ are nearly collinear, so $\Phi^\top\Phi$ is badly conditioned. Check the condition number.

In [ ]:
for d in (3, 9, 15):
    Phi = np.vander(X_tr["x"].to_numpy(), d + 1, increasing=True)
    print(f"degree {d:>2}: cond(Phi^T Phi) = {np.linalg.cond(Phi.T @ Phi):.1e}")

**Feature explosion.** Columns grow combinatorially with features and degree, so more data or stronger regularisation is needed as you add them.

## Summary

- Polynomial regression is linear regression on engineered columns. Nothing else changes.
- Training error always falls with degree. Test error falls, then rises. The best degree sits where the curve bottoms out, near the noise floor.
- High degrees need huge, cancelling weights, and they extrapolate badly.
- Choose the degree by cross-validation, or keep it flexible and add a penalty.

Next: `12_gradient_descent_variants.ipynb`, on how the minimum is found when the normal equation is too expensive.